# 🧠 Brain-Master — Worker de Inferencia en Colab (GPU T4 gratis)

Levanta el worker gRPC completo en la T4 de Colab y lo expone a internet:

- **gRPC (:50051) → túnel `bore`** (TCP puro, ideal para gRPC/HTTP2, sin registro)
- **Artefactos HTTP (:50052) → túnel `cloudflared`** (descarga de PNG/MP4)

Al final imprime dos líneas:

```
PYTHON_WORKER_HOST=bore.pub:<puerto>      # config del gateway Go
BM_WORKER_ARTIFACT_BASE=https://<tunel>   # config del gateway Go
```

Pégalas en tu máquina donde corre el gateway (o usa
`deploy/colab/connect-gateway.ps1`) y genera desde el front.

⚠️ La sesión free dura horas y al cerrar la pestaña se corta: re-ejecuta la
celda 4 para reconectar (los túneles cambian de puerto/URL).

## 1 — Verificar GPU

In [ ]:
# Runtime > Change runtime type > T4 GPU (imprescindible antes de ejecutar)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2 — Obtener el código

In [ ]:
import os
# Opción A: clonar el repo público (ya contiene inference-worker/).
#           Si el repo llegara a ser privado: sube inference-worker.zip
#           (opción B) o clona con token: https://<TU_TOKEN>@github.com/synappta4ai/brain-master.git
REPO_URL = "https://github.com/synappta4ai/brain-master.git"
# Opción B: subir la carpeta inference-worker.zip con el panel de archivos
#           y descomprimirla en /content/inference-worker

if os.path.exists("/content/inference-worker/server.py"):
    print("usando /content/inference-worker (subido a mano)")
else:
    !git clone --depth 1 $REPO_URL /content/brain-master
    !ln -s /content/brain-master/inference-worker /content/inference-worker
%cd /content/inference-worker

## 3 — Dependencias y túneles (bore + cloudflared)

In [ ]:
# Stack de IA (Colab ya trae torch+CUDA)
!pip install -q grpcio==1.84.0 grpcio-tools==1.84.0 "protobuf>=5.29.3" \
    "diffusers>=0.36.0" "transformers>=4.57.0" tokenizers accelerate safetensors \
    sentencepiece einops open_clip_torch imageio imageio-ffmpeg Pillow psutil

In [ ]:
# bore v0.6.0 (túnel TCP para gRPC) + cloudflared (túnel HTTP para artefactos)
!curl -sL -o /tmp/bore.tar.gz https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
!tar -xzf /tmp/bore.tar.gz -C /usr/local/bin && chmod +x /usr/local/bin/bore
!curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!bore --version && cloudflared --version

## 4 — Arrancar worker + túneles ⭐ (re-ejecutar para reconectar)

In [ ]:
import subprocess, time, re, os

os.environ["BM_OUTPUT_DIR"] = "/content/outputs"
os.makedirs("/content/outputs", exist_ok=True)

# Mata instancias previas (re-conexión)
!pkill -f 'python server.py' 2>/dev/null; pkill -f 'bore local' 2>/dev/null; pkill -f cloudflared 2>/dev/null
time.sleep(2)

worker = subprocess.Popen(["python", "server.py"],
                          stdout=open("/content/worker.log", "a"),
                          stderr=subprocess.STDOUT)
time.sleep(6)

def wait_for(path, pattern, tries=30):
    for _ in range(tries):
        time.sleep(1)
        m = re.search(pattern, open(path).read())
        if m:
            return m.group(0)
    raise RuntimeError(f"patrón no apareció en {path}:\n" + open(path).read())

# --- gRPC por bore (TCP puro). SIN --port fijo: bore.pub asigna uno libre
#     (pedir 50051 fijo colisiona con otros usuarios del relay público).
bore = subprocess.Popen(["bore", "local", "50051", "--to", "bore.pub"],
                        stdout=open("/content/bore.log", "w"), stderr=subprocess.STDOUT)
bore_port = wait_for("/content/bore.log", r"listening at bore\.pub:(\d+)", 30)

# --- Artefactos por cloudflared (HTTP)
cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:50052",
                       "--no-autoupdate"],
                      stdout=open("/content/cf.log", "w"), stderr=subprocess.STDOUT)
art_url = wait_for("/content/cf.log", r"https://[a-z0-9-]+\.trycloudflare\.com")

print("\n" + "=" * 64)
print("PYTHON_WORKER_HOST=bore.pub:" + bore_port)
print("BM_WORKER_ARTIFACT_BASE=" + art_url)
print("=" * 64)
print("Pega esas 2 líneas en la máquina del gateway (ver connect-gateway.ps1)")


## 5 — Smoke test local: difusión real en la T4

In [ ]:
# Primera generación REAL (descarga SD-Tiny-Test ~100MB la primera vez)
!python - <<'EOF'
import asyncio, grpc, sys
sys.path.insert(0, '.')
import inference_pb2, inference_pb2_grpc

async def main():
    async with grpc.aio.insecure_channel('127.0.0.1:50051') as ch:
        stub = inference_pb2_grpc.InferenceServiceStub(ch)
        cat = await stub.ListModels(inference_pb2.ListModelsRequest())
        print(f"catálogo: {len(cat.models)} modelos | cuda={cat.cuda_available} ({cat.device_name})")
        req = inference_pb2.MediaGenerationRequest(
            job_id='colab-smoke', mode='image', model_name='SD-Tiny-Test',
            prompt='a neon city at dusk, cinematic', steps=4, width=512, height=512)
        async for ev in stub.GenerateMedia(req):
            print(f"  {ev.status:14s} {ev.percentage:5.1f}%")

asyncio.run(main())
EOF

## 6 — Verificar los túneles desde fuera (como lo hará el gateway)

In [ ]:
# gRPC a través de bore.pub (el gateway usará exactamente esta ruta)
!python - <<'EOF'
import re, asyncio, grpc, sys
sys.path.insert(0, '.')
port = re.search(r'listening at bore\.pub:(\d+)', open('/content/bore.log').read()).group(1)
import inference_pb2, inference_pb2_grpc

async def main():
    async with grpc.aio.insecure_channel(f'bore.pub:{port}') as ch:
        stub = inference_pb2_grpc.InferenceServiceStub(ch)
        tel = await stub.GetGpuTelemetry(inference_pb2.GpuTelemetryRequest())
        print(f"✓ gRPC por bore.pub:{port} → device={tel.device_name}, VRAM libre={tel.free_vram_mb}MB")

asyncio.run(main())
EOF

# Artefacto vía cloudflared (el front usará esta URL)
!python - <<'EOF'
import re, requests
url = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read()).group(0)
r = requests.get(url + '/healthz', timeout=15)
print('✓ artifact server por túnel:', r.status_code, r.json())
EOF

## 7 — Conectar tu gateway (en tu máquina)

Windows (PowerShell):
```powershell
.\deploy\colab\connect-gateway.ps1 -WorkerHost "bore.pub:PORT" -ArtifactBase "https://xxx.trycloudflare.com"
```

Linux/macOS:
```bash
export PYTHON_WORKER_HOST=bore.pub:PORT BM_WORKER_ARTIFACT_BASE=https://xxx.trycloudflare.com
# reinicia el gateway con esas variables
```

In [ ]:
# Monitor del worker (detener con ■)
!tail -f /content/worker.log